# Predicting the Cause: What Drives Major Power Outages in the U.S.?

**Name(s)**: Shivam Sharma

**Website Link**: https://shivamsharma0608.github.io/power_outage_causation/

In [38]:
# Standard data manipulation and visualization libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Sklearn: pipeline tools, preprocessing, models, and evaluation metrics
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')

import os
pd.options.plotting.backend = 'plotly'
from dsc80_utils import *

!pip install openpyxl
!pip install folium
import folium
print(os.getcwd())
# Set repo path for saving plot assets, update if running on a different machine
REPO_PATH = '/Users/shivamsharma0608/Desktop/power_outage_causation'

/Users/shivamsharma0608/Desktop/power_outage_causation


## Step 1: Introduction

This project analyzes a dataset of **1,534 major power outages** that occurred 
in the continental U.S. between January 2000 and July 2016. Each row represents 
a single outage event, with information about its cause, location, duration, 
climate conditions, and the economic and demographic characteristics of the 
affected area.

**Central Question:** What characteristics — including location, climate, timing, 
and economic factors — are associated with each cause category of major power outage?

Understanding what drives different types of outages matters because utility 
companies, emergency responders, and policymakers must respond very differently 
depending on cause. A severe weather outage requires field repair crews; an 
intentional attack may require law enforcement and cybersecurity response. Being 
able to identify the likely cause quickly — using only information available at 
the moment an outage is detected — has real operational value.

The relevant columns for this analysis are:

| Column | Type | Description |
|---|---|---|
| `CAUSE.CATEGORY` | Nominal | Category of the event causing the outage (our target) |
| `CLIMATE.REGION` | Nominal | U.S. climate region where the outage occurred |
| `NERC.REGION` | Nominal | North American Electric Reliability Corporation region |
| `ANOMALY.LEVEL` | Quantitative | Oceanic El Niño/La Niña index at time of outage |
| `CLIMATE.CATEGORY` | Nominal | Climate episode: Warm, Cold, or Normal |
| `U.S._STATE` | Nominal | State where the outage occurred |
| `OUTAGE.START` | DateTime | Combined date and time the outage began |
| `OUTAGE.DURATION` | Quantitative | Duration of the outage in minutes |
| `CUSTOMERS.AFFECTED` | Quantitative | Number of customers affected |
| `POPPCT_URBAN` | Quantitative | Percentage of state population living in urban areas |
| `TOTAL.PRICE` | Quantitative | Average retail electricity price (cents/kWh) |
| `SEASON` | Nominal | Season derived from outage start date (engineered feature) |

In [39]:
# Load the Excel file — skip first 5 rows (metadata) and row 6 is header
# Drop the units row (index 0 after load) and the 'variables' artifact column
outages_raw = pd.read_excel('outage.xlsx', skiprows=5, header=0)
outages_raw = outages_raw.drop(index=0).reset_index(drop=True)
outages_raw = outages_raw.drop(columns=['variables'], errors='ignore')
print(f"Shape: {outages_raw.shape}")
print(outages_raw.head(3))

# List of columns relevant to our central question
relevant_cols = [
    'YEAR', 'MONTH', 'U.S._STATE', 'NERC.REGION', 'CLIMATE.REGION',
    'ANOMALY.LEVEL', 'CLIMATE.CATEGORY', 'CAUSE.CATEGORY',
    'CAUSE.CATEGORY.DETAIL', 'OUTAGE.DURATION', 'DEMAND.LOSS.MW',
    'CUSTOMERS.AFFECTED', 'TOTAL.PRICE', 'TOTAL.SALES', 'TOTAL.CUSTOMERS',
    'POPPCT_URBAN', 'POPDEN_URBAN', 'AREAPCT_URBAN',
    'OUTAGE.START.DATE', 'OUTAGE.START.TIME',
    'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME',
]
outages_raw[relevant_cols]

Shape: (1534, 56)
   OBS    YEAR  MONTH U.S._STATE  ... AREAPCT_UC PCT_LAND PCT_WATER_TOT  \
0  1.0  2011.0    7.0  Minnesota  ...        0.6    91.59          8.41   
1  2.0  2014.0    5.0  Minnesota  ...        0.6    91.59          8.41   
2  3.0  2010.0   10.0  Minnesota  ...        0.6    91.59          8.41   

  PCT_WATER_INLAND  
0             5.48  
1             5.48  
2             5.48  

[3 rows x 56 columns]


,YEAR,MONTH,U.S._STATE,NERC.REGION,...,OUTAGE.START.DATE,OUTAGE.START.TIME,OUTAGE.RESTORATION.DATE,OUTAGE.RESTORATION.TIME
0,2011.0,7.0,Minnesota,MRO,...,2011-07-01 00:00:00,17:00:00,2011-07-03 00:00:00,20:00:00
1,2014.0,5.0,Minnesota,MRO,...,2014-05-11 00:00:00,18:38:00,2014-05-11 00:00:00,18:39:00
2,2010.0,10.0,Minnesota,MRO,...,2010-10-26 00:00:00,20:00:00,2010-10-28 00:00:00,22:00:00
...,...,...,...,...,...,...,...,...,...
1531,2009.0,8.0,South Dakota,RFC,...,2009-08-29 00:00:00,22:54:00,2009-08-29 00:00:00,23:53:00
1532,2009.0,8.0,South Dakota,MRO,...,2009-08-29 00:00:00,11:00:00,2009-08-29 00:00:00,14:01:00
1533,2000.0,NaN,Alaska,ASCC,...,NaN,NaN,NaN,NaN


## Step 2: Data Cleaning and Exploratory Data Analysis

The raw Excel file required several cleaning steps before analysis:

**1. Dropping the units row:** The raw file contains a "units" row immediately 
after the header (e.g. "year", "month", "megawatt") that is not actual data. 
This was dropped to avoid corrupting numeric columns with string values.

**2. Merging timestamp columns:** The outage start date and time are stored as 
separate columns (`OUTAGE.START.DATE` and `OUTAGE.START.TIME`), reflecting how 
the data was originally entered into a spreadsheet. We combined these into a 
single `pd.Timestamp` column called `OUTAGE.START`, and did the same for 
`OUTAGE.RESTORATION`. This allows us to extract time-based features like season 
and hour of day directly from a single column.

**3. Replacing 0s with NaN in duration/impact columns:** `OUTAGE.DURATION`, 
`CUSTOMERS.AFFECTED`, and `DEMAND.LOSS.MW` each contain 0 values that represent 
missing or unreported data rather than true zeros — a power outage that affected 
0 customers or lasted 0 minutes is not a real event. Replacing these with NaN 
ensures they are excluded from aggregations. After this step, `OUTAGE.DURATION` 
has 136 missing values, `CUSTOMERS.AFFECTED` has 655, and `DEMAND.LOSS.MW` has 901.

**4. Engineering a SEASON column:** Month was mapped to one of four seasons 
(Winter, Spring, Summer, Fall) because cause categories like severe weather are 
strongly tied to seasonal weather patterns. Using season rather than raw month 
reduces dimensionality and makes the feature more interpretable. The distribution 
is: Summer (529), Winter (383), Spring (338), Fall (275).

**5. Filtering rows with missing CAUSE.CATEGORY:** Since our analysis centers on 
cause category, rows where this value is missing cannot contribute meaningfully 
to either EDA or modeling. No rows were removed by this step — CAUSE.CATEGORY 
has 0 missing values in this dataset.

In [ ]:
outages = outages_raw.copy()

# 1. Combine date + time into single pd.Timestamp columns (required by spec)
outages['OUTAGE.START'] = pd.to_datetime(
    outages['OUTAGE.START.DATE'].astype(str) + ' ' + outages['OUTAGE.START.TIME'].astype(str),
    errors='coerce'
)
outages['OUTAGE.RESTORATION'] = pd.to_datetime(
    outages['OUTAGE.RESTORATION.DATE'].astype(str) + ' ' + outages['OUTAGE.RESTORATION.TIME'].astype(str),
    errors='coerce'
)

# 2. Drop the original split date/time columns
outages = outages.drop(columns=[
    'OUTAGE.START.DATE', 'OUTAGE.START.TIME',
    'OUTAGE.RESTORATION.DATE', 'OUTAGE.RESTORATION.TIME'
])

# 3. Replace 0s in columns where 0 is meaningless with NaN
#    (e.g. 0 customers affected or 0 demand loss likely means data is missing)
for col in ['DEMAND.LOSS.MW', 'CUSTOMERS.AFFECTED', 'OUTAGE.DURATION']:
    outages[col] = outages[col].replace(0, np.nan)

# 4. Extract useful time features from the timestamp
outages['OUTAGE.MONTH'] = outages['OUTAGE.START'].dt.month
outages['OUTAGE.HOUR'] = outages['OUTAGE.START'].dt.hour

# 5. Create a SEASON column from month
def month_to_season(month):
    if pd.isna(month):
        return np.nan
    month = int(month)
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

outages['SEASON'] = outages['OUTAGE.MONTH'].apply(month_to_season)

# 6. Keep only rows where CAUSE.CATEGORY is not null (our target)
outages = outages[outages['CAUSE.CATEGORY'].notna()].reset_index(drop=True)

print(f"Cleaned shape: {outages.shape}")
print(f"\nCause category distribution:\n{outages['CAUSE.CATEGORY'].value_counts()}")
print(f"\nMissing values in key columns:")
print(outages[['CAUSE.CATEGORY','CLIMATE.REGION','ANOMALY.LEVEL',
               'CUSTOMERS.AFFECTED','DEMAND.LOSS.MW','OUTAGE.DURATION']].isna().sum())

# Show head for website
print(outages[['YEAR','U.S._STATE','CLIMATE.REGION','CAUSE.CATEGORY',
               'OUTAGE.DURATION','CUSTOMERS.AFFECTED','OUTAGE.START']].head())


# ── Univariate Analysis ───────────────────────────────────────────────────────

# Plot 1: Distribution of CAUSE.CATEGORY
cause_counts = outages['CAUSE.CATEGORY'].value_counts().reset_index()
cause_counts.columns = ['Cause Category', 'Count']

fig1 = px.bar(
    cause_counts,
    x='Cause Category', y='Count',
    title='Distribution of Power Outage Cause Categories',
    labels={'Cause Category': 'Cause Category', 'Count': 'Number of Outages'},
    color='Count',
    color_continuous_scale='Blues',
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig1.update_layout(showlegend=False, xaxis_tickangle=-30)
fig1.show()
fig1.write_html('assets/cause_distribution.html', include_plotlyjs='cdn')

# Plot 2: Distribution of OUTAGE.DURATION (log scale due to skew)
fig2 = px.histogram(
    outages[outages['OUTAGE.DURATION'].notna()],
    x='OUTAGE.DURATION',
    nbins=60,
    title='Distribution of Outage Duration (minutes)',
    labels={'OUTAGE.DURATION': 'Duration (minutes)'},
    template='plotly_white',
    color_discrete_sequence=['steelblue']
)
fig2.update_layout(yaxis_title='Count')
fig2.show()
fig2.write_html('assets/duration_distribution.html', include_plotlyjs='cdn')


# ── Bivariate Analysis ────────────────────────────────────────────────────────

# Plot 3: Average outage duration by cause category
duration_by_cause = (
    outages.groupby('CAUSE.CATEGORY')['OUTAGE.DURATION']
    .median()
    .reset_index()
    .sort_values('OUTAGE.DURATION', ascending=False)
)
duration_by_cause.columns = ['Cause Category', 'Median Duration (min)']

fig3 = px.bar(
    duration_by_cause,
    x='Cause Category', y='Median Duration (min)',
    title='Median Outage Duration by Cause Category',
    template='plotly_white',
    color='Median Duration (min)',
    color_continuous_scale='Reds',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.update_layout(xaxis_tickangle=-30, showlegend=False)
fig3.show()
fig3.write_html('assets/duration_by_cause.html', include_plotlyjs='cdn')

# Plot 4: Cause category breakdown by climate region (stacked bar)
climate_cause = (
    outages.groupby(['CLIMATE.REGION', 'CAUSE.CATEGORY'])
    .size()
    .reset_index(name='Count')
)
fig4 = px.bar(
    climate_cause,
    x='CLIMATE.REGION', y='Count',
    color='CAUSE.CATEGORY',
    title='Cause Category Breakdown by Climate Region',
    labels={'CLIMATE.REGION': 'Climate Region', 'Count': 'Number of Outages'},
    template='plotly_white',
    barmode='stack'
)
fig4.update_layout(xaxis_tickangle=-30, legend_title='Cause Category')
fig4.show()
fig4.write_html('assets/cause_by_climate.html', include_plotlyjs='cdn')

# Plot 5: Box plot — Anomaly Level distribution by Cause Category
# This shows whether certain causes are associated with more extreme climate conditions
fig_box = px.box(
    outages[outages['ANOMALY.LEVEL'].notna()],
    x='CAUSE.CATEGORY',
    y='ANOMALY.LEVEL',
    title='Climate Anomaly Level by Cause Category',
    labels={'CAUSE.CATEGORY': 'Cause Category', 'ANOMALY.LEVEL': 'Anomaly Level (El Niño/La Niña index)'},
    template='plotly_white',
    color='CAUSE.CATEGORY'
)
fig_box.update_layout(xaxis_tickangle=-30, showlegend=False)
fig_box.show()
fig_box.write_html(f'{REPO_PATH}/assets/anomaly_by_cause.html', include_plotlyjs='cdn')


# Count outages per state
state_counts = outages.groupby('U.S._STATE').size().reset_index(name='outage_count')
# Load US states GeoJSON
us_states_url = 'https://raw.githubusercontent.com/python-visualization/folium/master/examples/data/us-states.json'
# Create base map centered on US
m = folium.Map(location=[37.8, -96], zoom_start=4, tiles='CartoDB positron')
# Add choropleth layer
folium.Choropleth(
    geo_data=us_states_url,
    name='Power Outages by State',
    data=state_counts,
    columns=['U.S._STATE', 'outage_count'],
    key_on='feature.properties.name',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Number of Major Power Outages (2000-2016)',
    nan_fill_color='white'
).add_to(m)
folium.LayerControl().add_to(m)
m
REPO_PATH = '/Users/shivamsharma0608/Desktop/power_outage_causation'
with open(f'{REPO_PATH}/assets/outage_map.html', 'w') as f:
    f.write(m._repr_html_())


# ── Interesting Aggregates ────────────────────────────────────────────────────

# Pivot table: median customers affected and duration by cause + season
pivot = outages.pivot_table(
    index='CAUSE.CATEGORY',
    columns='SEASON',
    values='CUSTOMERS.AFFECTED',
    aggfunc='median'
)
print("\nMedian Customers Affected by Cause Category and Season:")
print(pivot.round(0))
print(pivot.round(0).to_markdown())  

Cleaned shape: (1534, 57)

Cause category distribution:
CAUSE.CATEGORY
severe weather                   763
intentional attack               418
system operability disruption    127
public appeal                     69
equipment failure                 60
fuel supply emergency             51
islanding                         46
Name: count, dtype: int64

Missing values in key columns:
CAUSE.CATEGORY          0
CLIMATE.REGION          6
ANOMALY.LEVEL           9
CUSTOMERS.AFFECTED    655
DEMAND.LOSS.MW        901
OUTAGE.DURATION       136
dtype: int64
     YEAR U.S._STATE      CLIMATE.REGION      CAUSE.CATEGORY  OUTAGE.DURATION  \
0  2011.0  Minnesota  East North Central      severe weather           3060.0   
1  2014.0  Minnesota  East North Central  intentional attack              1.0   
2  2010.0  Minnesota  East North Central      severe weather           3000.0   
3  2012.0  Minnesota  East North Central      severe weather           2550.0   
4  2015.0  Minnesota  East North Centr


Median Customers Affected by Cause Category and Season:
SEASON                             Fall    Spring    Summer    Winter
CAUSE.CATEGORY                                                       
equipment failure              900000.0   80915.0   45452.0   52000.0
fuel supply emergency               NaN       NaN       NaN       1.0
intentional attack               9200.0    5852.0    1100.0    2500.0
islanding                        7077.0    9700.0     606.0    6635.0
public appeal                       NaN       NaN    8000.0   18600.0
severe weather                 118000.0  102568.0  109000.0  118000.0
system operability disruption  104000.0   82500.0   33500.0   51982.0
| CAUSE.CATEGORY                |   Fall |   Spring |   Summer |   Winter |
|:------------------------------|-------:|---------:|---------:|---------:|
| equipment failure             | 900000 |    80915 |    45452 |    52000 |
| fuel supply emergency         |    nan |      nan |      nan |        1 |
| intenti

## Step 3: Assessment of Missingness

### MNAR Analysis

The `CAUSE.CATEGORY.DETAIL` column is likely **MNAR** (Missing Not at Random). 
Looking at the missingness rate by cause category reveals that `islanding` and 
`public appeal` have **100% missingness** in this column, while `intentional attack` 
has 11.5% and `severe weather` has 24.5%. This pattern cannot be explained by 
any other observed variable — the missingness is directly tied to the value of 
the cause category itself, which is the definition of MNAR. Operators likely 
do not record details for islanding events because they are routine and 
self-resolving, and for public appeal events because no specific incident 
detail exists. To make this MAR, we would need additional metadata such as 
NERC incident report classifications that could explain *why* details are 
omitted for certain cause types.

### Missingness Dependency

We analyze the missingness of `CUSTOMERS.AFFECTED`, which is missing in 
**655 of 1,534 rows (42.7%)**. This is non-trivial and shows a striking pattern 
by cause: intentional attack outages are missing customer counts **95.5%** of 
the time, while severe weather outages are missing only **7.2%** of the time. 
This makes the missingness in `CUSTOMERS.AFFECTED` likely **MAR** (dependent 
on `CAUSE.CATEGORY`, which is observed), not MCAR.

In [41]:
# ── MNAR Discussion (no code needed — reasoning only) ─────────────────────────
# See notebook markdown cell above for MNAR reasoning on CAUSE.CATEGORY.DETAIL

# ── Missingness Dependency Analysis ──────────────────────────────────────────
# Column chosen: CUSTOMERS.AFFECTED — missing in 655/1534 rows (42.7%)
# This is non-trivial and meaningful: the missingness varies hugely by cause type.
# Intentional attack: 95.5% missing | Severe weather: only 7.2% missing
# This pattern suggests utilities don't report customer counts for targeted attacks.

outages['CUSTOMERS.AFFECTED.MISSING'] = outages['CUSTOMERS.AFFECTED'].isna()
print(f"CUSTOMERS.AFFECTED missing rate: {outages['CUSTOMERS.AFFECTED.MISSING'].mean():.2%}")
print("\nMissing rate by CAUSE.CATEGORY:")
print(outages.groupby('CAUSE.CATEGORY')['CUSTOMERS.AFFECTED']
      .apply(lambda x: x.isna().mean()).round(3))

# --- Test 1: Does missingness depend on CAUSE.CATEGORY? ---
# Test statistic: TVD (CAUSE.CATEGORY is categorical)
missing_mask = outages['CUSTOMERS.AFFECTED.MISSING']

def tvd(group_a, group_b):
    """Total Variation Distance between two categorical distributions."""
    cats = set(list(group_a) + list(group_b))
    dist_a = pd.Series(group_a).value_counts(normalize=True)
    dist_b = pd.Series(group_b).value_counts(normalize=True)
    return sum(abs(dist_a.get(c, 0) - dist_b.get(c, 0)) for c in cats) / 2

observed_tvd = tvd(
    outages.loc[missing_mask, 'CAUSE.CATEGORY'],
    outages.loc[~missing_mask, 'CAUSE.CATEGORY']
)

n_permutations = 500
tvd_stats = []
for _ in range(n_permutations):
    shuffled = missing_mask.sample(frac=1).values
    tvd_stats.append(tvd(
        outages.loc[shuffled, 'CAUSE.CATEGORY'],
        outages.loc[~shuffled, 'CAUSE.CATEGORY']
    ))

p_val_cause = np.mean(np.array(tvd_stats) >= observed_tvd)
print(f"\nTest 1 — CAUSE.CATEGORY")
print(f"  Observed TVD: {observed_tvd:.4f}")
print(f"  p-value: {p_val_cause:.4f}")
print(f"  Conclusion: missingness of CUSTOMERS.AFFECTED DOES depend on CAUSE.CATEGORY")

# Plot A: Empirical distribution of TVD permutation test
fig5 = px.histogram(
    x=tvd_stats, nbins=40,
    title='Permutation Test: Missingness of CUSTOMERS.AFFECTED vs. CAUSE.CATEGORY',
    labels={'x': 'TVD Statistic'},
    template='plotly_white',
    color_discrete_sequence=['lightblue']
)
fig5.add_vline(x=observed_tvd, line_color='red', line_dash='dash',
               annotation_text=f'Observed TVD = {observed_tvd:.3f}',
               annotation_position='top right')
fig5.update_layout(yaxis_title='Count')
fig5.show()
fig5.write_html(f'{REPO_PATH}/assets/missingness_permtest.html', 
                include_plotlyjs='cdn')

# --- Test 2: Does missingness depend on ANOMALY.LEVEL? ---
# Test statistic: absolute difference in means (ANOMALY.LEVEL is numeric)
observed_diff = abs(
    outages.loc[missing_mask, 'ANOMALY.LEVEL'].mean() -
    outages.loc[~missing_mask, 'ANOMALY.LEVEL'].mean()
)

diff_stats = []
for _ in range(n_permutations):
    shuffled = missing_mask.sample(frac=1).values
    diff_stats.append(abs(
        outages.loc[shuffled, 'ANOMALY.LEVEL'].mean() -
        outages.loc[~shuffled, 'ANOMALY.LEVEL'].mean()
    ))

p_val_anomaly = np.mean(np.array(diff_stats) >= observed_diff)
print(f"\nTest 2 — ANOMALY.LEVEL")
print(f"  Observed |diff in means|: {observed_diff:.4f}")
print(f"  p-value: {p_val_anomaly:.4f}")
print(f"  Conclusion: missingness of CUSTOMERS.AFFECTED does NOT depend on ANOMALY.LEVEL")

# Plot B: Distribution of ANOMALY.LEVEL when CUSTOMERS.AFFECTED is missing vs. not
# (This is the Lecture 8 style plot — shows the actual distributional difference)
anomaly_plot_df = outages[outages['ANOMALY.LEVEL'].notna()].copy()
anomaly_plot_df['Customers Affected'] = anomaly_plot_df['CUSTOMERS.AFFECTED.MISSING'].map(
    {True: 'Missing', False: 'Not Missing'}
)

fig_dist = px.histogram(
    anomaly_plot_df,
    x='ANOMALY.LEVEL',
    color='Customers Affected',
    barmode='overlay',
    histnorm='probability',
    nbins=30,
    title='Distribution of Anomaly Level by Missingness of Customers Affected',
    labels={'ANOMALY.LEVEL': 'Anomaly Level (El Niño/La Niña index)'},
    template='plotly_white',
    color_discrete_map={'Missing': 'tomato', 'Not Missing': 'steelblue'},
    opacity=0.6
)
fig_dist.update_layout(yaxis_title='Proportion')
fig_dist.show()
fig_dist.write_html(f'{REPO_PATH}/assets/missingness_distribution.html', 
                    include_plotlyjs='cdn')

CUSTOMERS.AFFECTED missing rate: 42.70%

Missing rate by CAUSE.CATEGORY:
CAUSE.CATEGORY
equipment failure                0.52
fuel supply emergency            0.98
intentional attack               0.95
islanding                        0.37
public appeal                    0.85
severe weather                   0.07
system operability disruption    0.35
Name: CUSTOMERS.AFFECTED, dtype: float64

Test 1 — CAUSE.CATEGORY
  Observed TVD: 0.7558
  p-value: 0.0000
  Conclusion: missingness of CUSTOMERS.AFFECTED DOES depend on CAUSE.CATEGORY



Test 2 — ANOMALY.LEVEL
  Observed |diff in means|: 0.0497
  p-value: 0.2160
  Conclusion: missingness of CUSTOMERS.AFFECTED does NOT depend on ANOMALY.LEVEL


## Step 4: Hypothesis Testing

**Question:** Do severe weather outages last significantly longer than intentional 
attack outages?

This question directly relates to our central theme — understanding what 
characteristics differ across cause categories. Duration is one of the most 
operationally important characteristics of an outage, and if cause categories 
systematically differ in duration, that strengthens the case that cause is a 
meaningful predictor of outage behavior.

**Null Hypothesis:** The distribution of outage durations is the same for severe 
weather outages and intentional attack outages. Any observed difference in median 
duration is due to random chance.

**Alternative Hypothesis:** Severe weather outages have longer durations on 
average than intentional attack outages.

**Test Statistic:** Difference in group medians (severe weather median minus 
intentional attack median). We use median rather than mean because 
`OUTAGE.DURATION` is heavily right-skewed with extreme outliers, making the 
median a more robust measure of center. We use a directional (one-sided) test 
because our alternative hypothesis specifically predicts severe weather lasts 
*longer* — physical infrastructure damage from storms takes more time to repair 
than targeted disruptions.

**Significance Level:** 0.05 — the standard threshold for social and data 
science research, which balances the risk of false positives against the cost 
of missing a real effect.

In [42]:
# Filter to the two cause categories of interest, drop missing durations
subset = outages[outages['CAUSE.CATEGORY'].isin(['severe weather', 'intentional attack'])]
subset = subset[subset['OUTAGE.DURATION'].notna()].reset_index(drop=True)

print(f"Sample sizes:")
print(f"  Severe weather:    {(subset['CAUSE.CATEGORY']=='severe weather').sum()} outages")
print(f"  Intentional attack: {(subset['CAUSE.CATEGORY']=='intentional attack').sum()} outages")

sw_median = subset[subset['CAUSE.CATEGORY'] == 'severe weather']['OUTAGE.DURATION'].median()
ia_median = subset[subset['CAUSE.CATEGORY'] == 'intentional attack']['OUTAGE.DURATION'].median()
observed_diff_hyp = sw_median - ia_median

print(f"\nSevere weather median duration:    {sw_median:,.0f} minutes")
print(f"Intentional attack median duration: {ia_median:,.0f} minutes")
print(f"Observed difference in medians:     {observed_diff_hyp:,.0f} minutes")

# Permutation test: shuffle cause labels, recompute difference in medians
np.random.seed(42)
perm_diffs = []
for _ in range(1000):
    shuffled_labels = subset['CAUSE.CATEGORY'].sample(frac=1).values
    diff = (
        subset.loc[shuffled_labels == 'severe weather', 'OUTAGE.DURATION'].median() -
        subset.loc[shuffled_labels == 'intentional attack', 'OUTAGE.DURATION'].median()
    )
    perm_diffs.append(diff)

p_value_hyp = np.mean(np.array(perm_diffs) >= observed_diff_hyp)
print(f"\np-value: {p_value_hyp:.4f}")

# Plot
fig6 = px.histogram(
    x=perm_diffs, nbins=50,
    title='Permutation Test: Outage Duration — Severe Weather vs. Intentional Attack',
    labels={'x': 'Difference in Median Duration (minutes)'},
    template='plotly_white',
    color_discrete_sequence=['lightgreen']
)
fig6.add_vline(x=observed_diff_hyp, line_color='red', line_dash='dash',
               annotation_text=f'Observed = {observed_diff_hyp:,.0f} min',
               annotation_position='top right')
fig6.update_layout(yaxis_title='Count')
fig6.show()
fig6.write_html(f'{REPO_PATH}/assets/hypothesis_test.html', 
                include_plotlyjs='cdn')

Sample sizes:
  Severe weather:    741 outages
  Intentional attack: 332 outages

Severe weather median duration:    2,464 minutes
Intentional attack median duration: 92 minutes
Observed difference in medians:     2,372 minutes

p-value: 0.0000


## Step 5: Framing a Prediction Problem

### Prediction Problem

We aim to predict the **cause category (`CAUSE.CATEGORY`)** of a major power 
outage.

**Type:** Multiclass classification (7 classes: severe weather, intentional 
attack, system operability disruption, public appeal, equipment failure, fuel 
supply emergency, islanding)

**Response variable:** `CAUSE.CATEGORY`  
We chose this variable because it sits at the center of our entire EDA question — 
understanding what characteristics are associated with each cause. Building a 
classifier makes that question concrete and operational: if we can predict cause 
from features observable at the start of an outage, utility companies can dispatch 
the right response teams immediately rather than waiting for a manual assessment.

---

### Evaluation Metric: Weighted F1-Score

We evaluate using **weighted F1-score** rather than accuracy or macro F1.

The class distribution is highly imbalanced:
- Severe weather: ~50% of outages
- Intentional attack: ~27%
- All other 5 classes: ~23% combined

**Why not accuracy?** A classifier that always predicts "severe weather" would 
achieve ~50% accuracy while being completely useless for every other class. 
Accuracy rewards majority-class performance and masks failure on minority classes.

**Why not macro F1?** Macro F1 weights all classes equally regardless of how 
many examples they have. A class like "islanding" (46 examples) would count 
just as much as "severe weather" (763 examples), which is misleading for a 
real-world deployment where most outages are severe weather.

**Why weighted F1?** Weighted F1 computes F1 for each class separately and 
takes a weighted average by class support (number of true instances). This 
accounts for imbalance while still penalizing poor performance on any class 
proportionally to how often that class appears.

---

### Time of Prediction Justification

We are predicting cause **at the moment an outage is first detected** — before 
any response has occurred and before the outage has ended. This means we can 
only use features that are observable at that instant.

| Feature | Available at time of prediction? | Reason |
|---|---|---|
| `CLIMATE.REGION` | ✅ Yes | Static geographic property of the state |
| `NERC.REGION` | ✅ Yes | Static geographic property of the state |
| `ANOMALY.LEVEL` | ✅ Yes | Published monthly climate index, known in advance |
| `SEASON` | ✅ Yes | Derived from outage start timestamp |
| `POPPCT_URBAN` | ✅ Yes | Census demographic data, static |
| `TOTAL.PRICE` | ✅ Yes | Monthly electricity price, published in advance |
| `OUTAGE.DURATION` | ❌ No | Only known after outage ends |
| `CUSTOMERS.AFFECTED` | ❌ No | Only known after outage ends |
| `DEMAND.LOSS.MW` | ❌ No | Only known after outage ends |

In [43]:
# Class distribution of target variable
print("Class distribution of CAUSE.CATEGORY:")
class_dist = outages['CAUSE.CATEGORY'].value_counts()
class_dist_pct = outages['CAUSE.CATEGORY'].value_counts(normalize=True).round(3)
print(pd.DataFrame({'Count': class_dist, 'Proportion': class_dist_pct}))

# Majority class baseline — what accuracy would a naive classifier achieve?
majority_class_acc = class_dist_pct.iloc[0]
print(f"\nMajority class baseline accuracy (always predict 'severe weather'): {majority_class_acc:.1%}")
print("This is why we use weighted F1 instead of accuracy.")

Class distribution of CAUSE.CATEGORY:
                               Count  Proportion
CAUSE.CATEGORY                                  
severe weather                   763        0.50
intentional attack               418        0.27
system operability disruption    127        0.08
public appeal                     69        0.04
equipment failure                 60        0.04
fuel supply emergency             51        0.03
islanding                         46        0.03

Majority class baseline accuracy (always predict 'severe weather'): 49.7%
This is why we use weighted F1 instead of accuracy.


## Step 6: Baseline Model

Our baseline model predicts `CAUSE.CATEGORY` using **two features**:

| Feature | Type | Encoding |
|---|---|---|
| `CLIMATE.REGION` | Nominal | One-Hot Encoding (`OneHotEncoder`) — since climate region has no natural ordering, we encode each of the 9 regions as a binary indicator column |
| `ANOMALY.LEVEL` | Quantitative | Standard Scaling (`StandardScaler`) — we standardize to zero mean and unit variance to help Logistic Regression converge faster and treat all features on equal footing |

**Model:** Logistic Regression — a simple linear classifier that is a natural 
baseline for multiclass classification. It requires minimal hyperparameter tuning 
and gives us a performance floor to beat.

All steps (encoding + model training) are implemented in a single `sklearn` 
`Pipeline` to prevent data leakage between train and test sets.

In [44]:
baseline_features = ['CLIMATE.REGION', 'ANOMALY.LEVEL']
target = 'CAUSE.CATEGORY'

# Drop rows missing any feature or target
model_df = outages[baseline_features + [target]].dropna().reset_index(drop=True)
X = model_df[baseline_features]
y = model_df[target]

# Stratified split to preserve class proportions in both train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

# Build pipeline: encode CLIMATE.REGION, scale ANOMALY.LEVEL, then classify
baseline_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['CLIMATE.REGION']),
    ('num', StandardScaler(), ['ANOMALY.LEVEL']),
])
baseline_pipeline = Pipeline([
    ('preprocessor', baseline_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

baseline_pipeline.fit(X_train, y_train)

# Evaluate on BOTH train and test using our chosen metric (weighted F1)
train_f1 = f1_score(y_train, baseline_pipeline.predict(X_train), average='weighted')
test_f1  = f1_score(y_test,  baseline_pipeline.predict(X_test),  average='weighted')
train_acc = accuracy_score(y_train, baseline_pipeline.predict(X_train))
test_acc  = accuracy_score(y_test,  baseline_pipeline.predict(X_test))

print(f"\nBaseline Model Performance:")
print(f"  Train Weighted F1: {train_f1:.4f}")
print(f"  Test  Weighted F1: {test_f1:.4f}")
print(f"  Train Accuracy:    {train_acc:.4f}")
print(f"  Test  Accuracy:    {test_acc:.4f}")

# For context: majority class baseline
majority_f1 = f1_score(y_test, ['severe weather'] * len(y_test), average='weighted')
print(f"\nMajority class baseline F1: {majority_f1:.4f}")
print(f"Our improvement over majority class: {test_f1 - majority_f1:+.4f}")

# What classes does the model actually predict?
print(f"\nClasses predicted on test set:")
print(pd.Series(baseline_pipeline.predict(X_test)).value_counts())

Train size: 1216 | Test size: 304

Baseline Model Performance:
  Train Weighted F1: 0.4761
  Test  Weighted F1: 0.4951
  Train Accuracy:    0.5683
  Test  Accuracy:    0.5921

Majority class baseline F1: 0.3297
Our improvement over majority class: +0.1654

Classes predicted on test set:
severe weather        255
intentional attack     49
Name: count, dtype: int64


### Is this a good model?

**No** — a weighted F1 of 0.4951 on the test set is not good performance for 
a 7-class classifier, though it meaningfully beats the majority-class baseline 
of 0.3297 (+0.165).

The key weaknesses are:

1. **Only predicts 2 of 7 classes.** On the test set, the model exclusively 
predicts "severe weather" and "intentional attack" — the two most common classes. 
It completely fails on equipment failure, fuel supply emergency, islanding, 
public appeal, and system operability disruption.

2. **Only 2 features.** `CLIMATE.REGION` and `ANOMALY.LEVEL` alone do not 
capture enough information to distinguish 7 cause categories. For example, 
intentional attacks and equipment failures can occur in the same climate region 
under the same anomaly conditions.

3. **Linear decision boundary.** Logistic Regression assumes linear separability, 
but the relationship between geographic/climate features and cause category is 
likely non-linear.

The train F1 (0.4761) and test F1 (0.4951) are very close, indicating the model 
is not overfitting — it is simply underfitting due to insufficient features and 
model complexity. This gives us a clear direction for improvement in Step 7.

## Step 7: Final Model

### New Features

We add four features to the baseline's two (`CLIMATE.REGION`, `ANOMALY.LEVEL`), 
for six total. Each new feature is motivated by the data generating process:

| Feature | Type | Encoding | Motivation |
|---|---|---|---|
| `NERC.REGION` | Nominal | One-Hot Encoding | Different NERC regions have different grid infrastructure ages, regulatory environments, and attack histories. The WECC region (Western U.S.) has disproportionately more intentional attacks than other regions — information `CLIMATE.REGION` alone doesn't capture |
| `SEASON` | Nominal | One-Hot Encoding | Severe weather causes spike in summer (hurricanes, thunderstorms) and winter (ice storms). Equipment failures increase in summer due to heat stress on transformers. Season encodes this temporal pattern more cleanly than raw month |
| `POPPCT_URBAN` | Quantitative | Standard Scaling | Urban areas concentrate grid infrastructure, making them more likely targets of intentional attacks and system operability disruptions. Rural areas experience more weather-driven outages due to exposed lines |
| `TOTAL.PRICE` | Quantitative | Standard Scaling | Electricity price reflects the economic and infrastructure maturity of a region. Areas with high prices often have aging infrastructure prone to equipment failure; areas dependent on volatile fuel sources are more susceptible to fuel supply emergencies |

### Modeling Algorithm

We use a **Random Forest classifier** rather than Logistic Regression for three 
reasons grounded in the data generating process:

1. **Non-linear interactions.** The relationship between cause and features is 
not linear — e.g. intentional attacks are common in the West (NERC: WECC) in 
summer but not in the Northeast in winter. Random Forest naturally captures 
these interaction effects through deep decision trees.

2. **Mixed feature types.** Our features include both nominal (one-hot encoded) 
and quantitative columns. Random Forest handles high-dimensional sparse inputs 
(from OHE) well without being sensitive to feature scaling.

3. **Robustness to class imbalance.** Each tree in the forest sees a bootstrapped 
sample, which reduces the dominance of the majority class compared to a single 
linear boundary.

### Hyperparameter Tuning Plan

Before running GridSearchCV, we identify three hyperparameters to tune:

- **`n_estimators`** (100 vs 200): More trees reduce variance at the cost of 
compute time. We test both to see if additional trees improve stability.
- **`max_depth`** (None vs 10 vs 20): Unlimited depth risks overfitting on 
training data. Restricting depth regularizes the model by preventing trees from 
memorizing noise in minority classes.
- **`min_samples_split`** (2 vs 5): Higher values prevent splits on very small 
groups, reducing overfitting on rare classes like islanding (9 test examples).

We use `GridSearchCV` with 5-fold cross-validation scored on weighted F1 — the 
same metric we chose in Step 5 — so hyperparameter selection is directly 
optimizing for what we care about.

**Important:** To ensure a fair comparison with the baseline model, both models 
are trained and evaluated on the **exact same train/test split**, defined by 
dropping rows with any missing values across all final model features.

In [45]:
# Define the shared dataset — used for BOTH baseline and final model
# This ensures train/test splits are identical for a fair comparison (Note 1)
all_features = ['CLIMATE.REGION', 'NERC.REGION', 'SEASON',
                'ANOMALY.LEVEL', 'POPPCT_URBAN', 'TOTAL.PRICE']
baseline_features = ['CLIMATE.REGION', 'ANOMALY.LEVEL']
target = 'CAUSE.CATEGORY'

shared_df = outages[all_features + [target]].dropna().reset_index(drop=True)
X_all = shared_df[all_features]
y_all = shared_df[target]

# One shared split for both models — same random_state and stratify as before
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
print(f"Shared split — Train: {len(X_train)}, Test: {len(X_test)}")

# ── Re-run baseline on shared split for valid comparison ──────────────────────
baseline_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['CLIMATE.REGION']),
    ('num', StandardScaler(), ['ANOMALY.LEVEL']),
], remainder='drop')

baseline_pipeline = Pipeline([
    ('preprocessor', baseline_preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
baseline_pipeline.fit(X_train[baseline_features], y_train)
baseline_train_f1 = f1_score(y_train, baseline_pipeline.predict(X_train[baseline_features]), average='weighted')
baseline_test_f1  = f1_score(y_test,  baseline_pipeline.predict(X_test[baseline_features]),  average='weighted')
print(f"\nBaseline (on shared split):")
print(f"  Train F1: {baseline_train_f1:.4f} | Test F1: {baseline_test_f1:.4f}")

# ── Final model ───────────────────────────────────────────────────────────────
cat_feats = ['CLIMATE.REGION', 'NERC.REGION', 'SEASON']
num_feats  = ['ANOMALY.LEVEL', 'POPPCT_URBAN', 'TOTAL.PRICE']

final_preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_feats),
    ('num', StandardScaler(), num_feats),
])
final_pipeline = Pipeline([
    ('preprocessor', final_preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# GridSearchCV on training data only — test set remains unseen
param_grid = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5],
}
grid_search = GridSearchCV(
    final_pipeline, param_grid,
    cv=5, scoring='f1_weighted', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f"\nBest hyperparameters: {grid_search.best_params_}")
print(f"Best CV F1 (weighted): {grid_search.best_score_:.4f}")

# Evaluate best model on same train/test split
best_model = grid_search.best_estimator_
final_train_f1 = f1_score(y_train, best_model.predict(X_train), average='weighted')
final_test_f1  = f1_score(y_test,  best_model.predict(X_test),  average='weighted')
final_test_acc = accuracy_score(y_test, best_model.predict(X_test))

print(f"\nFinal Model:")
print(f"  Train Weighted F1: {final_train_f1:.4f}")
print(f"  Test  Weighted F1: {final_test_f1:.4f}")
print(f"  Test  Accuracy:    {final_test_acc:.4f}")
print(f"\nComparison on identical test set:")
print(f"  Baseline Test F1: {baseline_test_f1:.4f}")
print(f"  Final    Test F1: {final_test_f1:.4f}")
print(f"  Improvement:      {final_test_f1 - baseline_test_f1:+.4f}")

# Per-class breakdown
from sklearn.metrics import classification_report
print("\nClassification Report (Final Model):")
print(classification_report(y_test, best_model.predict(X_test)))

# Note on overfitting
print(f"Note: Train-Test F1 gap = {final_train_f1 - final_test_f1:.4f}")
print("The model overfits somewhat — max_depth=None allows fully grown trees.")
print("The gap is acceptable given the improvement over baseline on test data.")

# Confusion matrix
cm = confusion_matrix(y_test, best_model.predict(X_test), labels=best_model.classes_)
fig_cm = px.imshow(
    cm,
    x=best_model.classes_, y=best_model.classes_,
    labels=dict(x='Predicted', y='Actual', color='Count'),
    title='Confusion Matrix — Final Model',
    color_continuous_scale='Blues',
    template='plotly_white',
    text_auto=True
)
fig_cm.update_layout(xaxis_tickangle=-30)
fig_cm.show()
fig_cm.write_html(f'{REPO_PATH}/assets/confusion_matrix.html',
                  include_plotlyjs='cdn')

Shared split — Train: 1205, Test: 302

Baseline (on shared split):
  Train F1: 0.4763 | Test F1: 0.4861
Fitting 5 folds for each of 12 candidates, totalling 60 fits

Best hyperparameters: {'classifier__max_depth': None, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 200}
Best CV F1 (weighted): 0.6328

Final Model:
  Train Weighted F1: 0.9211
  Test  Weighted F1: 0.6420
  Test  Accuracy:    0.6656

Comparison on identical test set:
  Baseline Test F1: 0.4861
  Final    Test F1: 0.6420
  Improvement:      +0.1559

Classification Report (Final Model):
                               precision    recall  f1-score   support

            equipment failure       0.38      0.25      0.30        12
        fuel supply emergency       0.00      0.00      0.00        10
           intentional attack       0.73      0.73      0.73        83
                    islanding       0.00      0.00      0.00         9
                public appeal       0.60      0.43      0.50        14
 

### Discussion

**Improvement:** The final model achieves a test weighted F1 of **0.6420** vs. 
the baseline's **0.4861** on the same test set — an improvement of **+0.1559**. 
Unlike the baseline which only predicted 2 of 7 classes, the final model 
predicts all 7 classes, with strong performance on severe weather (F1=0.79) 
and intentional attack (F1=0.73).

**Overfitting:** The train F1 (0.9211) is substantially higher than the test F1 
(0.6420), a gap of 0.279. This occurs because `max_depth=None` allows trees to 
grow until leaves are pure, memorizing training patterns. Despite this, the test 
F1 still substantially beats the baseline, and the CV F1 of 0.6329 confirms the 
model generalizes meaningfully beyond training data.

**Remaining challenges:** Minority classes like fuel supply emergency (F1=0.00), 
islanding (F1=0.00), and system operability disruption (F1=0.20) are still 
difficult to predict. These classes are rare (< 50 test examples each) and may 
require additional features — such as grid infrastructure age or weather severity 
indices — that are not available in this dataset.

## Step 8: Fairness Analysis

We ask: **does our final model perform equally well for outages in highly 
urbanized states vs. less urbanized states?**

This is a meaningful fairness question because urbanization is directly tied to 
the types of outages a state experiences — high-urbanization states have more 
intentional attacks (a harder class to predict), while low-urbanization states 
are more dominated by severe weather (the easiest class). If our model is 
systematically worse for one group, it may be less useful for utility companies 
operating in those areas.

**Group X (High urbanization):** States where `POPPCT_URBAN` ≥ median (84.05%)  
**Group Y (Low urbanization):** States where `POPPCT_URBAN` < median (84.05%)

**Evaluation Metric:** Weighted F1-score — the same metric used throughout 
Steps 6 and 7, which accounts for class imbalance.

**Null Hypothesis:** Our model is fair. Its weighted F1 for high-urbanization 
and low-urbanization states are roughly the same, and any differences are due 
to random chance.

**Alternative Hypothesis:** Our model is unfair. Its weighted F1 for 
high-urbanization states is lower than for low-urbanization states.

**Test Statistic:** Absolute difference in weighted F1 scores between the two 
groups. We use absolute difference because our alternative hypothesis is 
two-sided — we are testing whether the model performs differently, not 
specifically worse in one direction.

**Significance Level:** 0.05

In [46]:
# Use X_test and y_test from the shared split defined in Step 7
# We do NOT modify best_model — we only evaluate it on subgroups of the test set
eval_df = X_test.copy()
eval_df['true_label'] = y_test.values
eval_df['pred_label'] = best_model.predict(X_test)

# Define groups by POPPCT_URBAN median
urban_median = eval_df['POPPCT_URBAN'].median()
high_urban = eval_df['POPPCT_URBAN'] >= urban_median
low_urban  = eval_df['POPPCT_URBAN'] <  urban_median

print(f"Urban median threshold: {urban_median:.2f}%")
print(f"High urbanization group: {high_urban.sum()} outages")
print(f"Low urbanization group:  {low_urban.sum()} outages")

# Observed F1 for each group
f1_high = f1_score(eval_df.loc[high_urban, 'true_label'],
                   eval_df.loc[high_urban, 'pred_label'],
                   average='weighted')
f1_low  = f1_score(eval_df.loc[low_urban, 'true_label'],
                   eval_df.loc[low_urban, 'pred_label'],
                   average='weighted')
observed_diff_fair = abs(f1_high - f1_low)

print(f"\nF1 (high urbanization): {f1_high:.4f}")
print(f"F1 (low urbanization):  {f1_low:.4f}")
print(f"Observed |difference|:  {observed_diff_fair:.4f}")

# Permutation test: shuffle POPPCT_URBAN values to randomly reassign group membership
np.random.seed(42)
fair_diffs = []
for _ in range(1000):
    # Shuffle the urbanization values to break any real group structure
    shuffled_urban = eval_df['POPPCT_URBAN'].sample(frac=1).values >= urban_median
    fa = f1_score(eval_df.loc[shuffled_urban, 'true_label'],
                  eval_df.loc[shuffled_urban, 'pred_label'],
                  average='weighted')
    fb = f1_score(eval_df.loc[~shuffled_urban, 'true_label'],
                  eval_df.loc[~shuffled_urban, 'pred_label'],
                  average='weighted')
    fair_diffs.append(abs(fa - fb))

p_value_fair = np.mean(np.array(fair_diffs) >= observed_diff_fair)
print(f"\np-value: {p_value_fair:.4f}")

# Plot empirical distribution of permutation test
fig7 = px.histogram(
    x=fair_diffs, nbins=40,
    title='Fairness Permutation Test: High vs. Low Urbanization States',
    labels={'x': 'Absolute Difference in Weighted F1'},
    template='plotly_white',
    color_discrete_sequence=['plum']
)
fig7.add_vline(x=observed_diff_fair, line_color='red', line_dash='dash',
               annotation_text=f'Observed = {observed_diff_fair:.3f}',
               annotation_position='top right')
fig7.update_layout(yaxis_title='Count')
fig7.show()
fig7.write_html(f'{REPO_PATH}/assets/fairness_test.html',
                include_plotlyjs='cdn')

Urban median threshold: 84.05%
High urbanization group: 159 outages
Low urbanization group:  143 outages

F1 (high urbanization): 0.5704
F1 (low urbanization):  0.7198
Observed |difference|:  0.1494

p-value: 0.0160


### Conclusion

With a p-value of **0.016**, which is less than our significance level of 0.05, 
we **reject the null hypothesis**. The evidence suggests our model does not 
perform equally across urbanization groups — it achieves a weighted F1 of 
**0.7198** for low-urbanization states but only **0.5704** for high-urbanization 
states, a difference of 0.1494 that is unlikely to be due to random chance.

This disparity likely arises from class composition differences between the groups. 
High-urbanization states have a more diverse mix of cause categories — including 
proportionally more intentional attacks, system operability disruptions, and fuel 
supply emergencies — all of which our model struggles to classify. Low-urbanization 
states are more dominated by severe weather, which our model predicts well (F1=0.79).

Importantly, we cannot conclude this is a causal relationship — it may reflect 
structural differences in what types of outages occur in urban vs. rural areas, 
rather than the model being biased against one group per se. Future work could 
address this by incorporating features more specific to urban grid infrastructure, 
such as grid age, substation density, or cybersecurity incident history.
